In [1]:

import json
import pickle
import time
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

start = time.time()

# ----------------------------
# 0) Project / MLflow setup
# ----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
PREP_ROOT = DATA_ROOT / "prepared"
REPORT_ROOT = PROJECT_ROOT / "reports"
ARTIFACT_ROOT = PROJECT_ROOT / "models" / "artifacts"
MLRUNS_ROOT = PROJECT_ROOT / "mlruns"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MLRUNS_ROOT.mkdir(parents=True, exist_ok=True)

tracking_uri = f"file:///{MLRUNS_ROOT.resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("recomart_svd_baseline")

FILE_PATH = PREP_ROOT / "interactions_prepared.csv"

# ----------------------------
# 1) Load prepared interactions
# ----------------------------
use_cols = ["user_id", "item_id", "event_weight"]
df = pd.read_csv(FILE_PATH, usecols=use_cols)

required_cols = {"user_id", "item_id", "event_weight"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded file:", FILE_PATH)
print("Raw shape:", df.shape)

# ----------------------------
# 2) Aggregate + reduce dataset
# ----------------------------
df = (
    df.groupby(["user_id", "item_id"], as_index=False, sort=False)["event_weight"]
      .sum()
)

MIN_USER_ITEMS = 5
MIN_ITEM_USERS = 20
MAX_USERS = 50000
MAX_ITEMS = 5000

changed = True
while changed:
    before = len(df)

    user_counts = df.groupby("user_id")["item_id"].nunique()
    keep_users = user_counts[user_counts >= MIN_USER_ITEMS].index
    df = df[df["user_id"].isin(keep_users)]

    item_counts = df.groupby("item_id")["user_id"].nunique()
    keep_items = item_counts[item_counts >= MIN_ITEM_USERS].index
    df = df[df["item_id"].isin(keep_items)]

    changed = len(df) != before

top_users = (
    df.groupby("user_id")["item_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(MAX_USERS)
    .index
)
df = df[df["user_id"].isin(top_users)]

top_items = (
    df.groupby("item_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(MAX_ITEMS)
    .index
)
df = df[df["item_id"].isin(top_items)].copy()

if df.empty:
    raise ValueError("Filtered dataset is empty. Relax thresholds.")

print("Filtered shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print("Unique items:", df["item_id"].nunique())

# ----------------------------
# 3) Leave-one-out split
# ----------------------------
rand = np.random.RandomState(42)
df["_rand"] = rand.rand(len(df))
test_idx = df.groupby("user_id")["_rand"].idxmin()

test_df = df.loc[test_idx, ["user_id", "item_id", "event_weight"]].copy()
train_df = df.drop(index=test_idx).copy()

train_counts = train_df.groupby("user_id")["item_id"].size()
valid_users = train_counts[train_counts >= 1].index
train_df = train_df[train_df["user_id"].isin(valid_users)].copy()
test_df = test_df[test_df["user_id"].isin(valid_users)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ----------------------------
# 4) Build train matrix
# ----------------------------
user_ids = train_df["user_id"].unique()
item_ids = train_df["item_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = np.array(item_ids)

test_df = test_df[
    test_df["user_id"].isin(user_to_idx) & test_df["item_id"].isin(item_to_idx)
].copy()

rows = train_df["user_id"].map(user_to_idx).to_numpy()
cols = train_df["item_id"].map(item_to_idx).to_numpy()
vals = train_df["event_weight"].astype(np.float32).to_numpy()

user_item = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_to_idx), len(item_to_idx)),
    dtype=np.float32
)

print("User-item matrix shape:", user_item.shape)

seen_by_user = train_df.groupby("user_id")["item_id"].apply(list).to_dict()
seen_idx_by_user = {
    u: np.array([item_to_idx[i] for i in items if i in item_to_idx], dtype=np.int32)
    for u, items in seen_by_user.items()
}

# ----------------------------
# 5) Train SVD
# ----------------------------
min_dim = min(user_item.shape)
n_components = min(20, max(2, min_dim - 1))

svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item).astype(np.float32)
item_factors = svd.components_.astype(np.float32)

print("Chosen n_components:", n_components)
print("Explained variance ratio sum:", round(svd.explained_variance_ratio_.sum(), 4))

# ----------------------------
# 6) Recommend fast
# ----------------------------
def recommend_svd(user_id, k=10):
    if user_id not in user_to_idx:
        return []

    uidx = user_to_idx[user_id]
    scores = user_factors[uidx] @ item_factors

    seen = seen_idx_by_user.get(user_id)
    if seen is not None and len(seen) > 0:
        scores[seen] = -np.inf

    top_n = min(k, len(scores))
    top_idx = np.argpartition(scores, -top_n)[-top_n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    return idx_to_item[top_idx].tolist()

# ----------------------------
# 7) Ranking metrics
# ----------------------------
def evaluate_ranking_metrics(test_df, recommend_fn, k=10, max_eval_users=10000):
    user_truth = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    eval_users = list(user_truth.keys())[:max_eval_users]

    precisions, recalls, ndcgs = [], [], []

    for user_id in eval_users:
        true_items = user_truth[user_id]
        recs = recommend_fn(user_id, k=k)
        if not recs:
            continue

        hits = np.array([1 if item in true_items else 0 for item in recs], dtype=np.float32)
        hit_count = hits.sum()

        precision = hit_count / k
        recall = hit_count / len(true_items)

        discounts = 1.0 / np.log2(np.arange(2, len(hits) + 2))
        dcg = float((hits * discounts).sum())

        ideal_len = min(len(true_items), k)
        idcg = float((np.ones(ideal_len) / np.log2(np.arange(2, ideal_len + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"precision_at_{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall_at_{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"ndcg_at_{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"evaluated_users_at_{k}": int(len(precisions)),
    }

metrics_5 = evaluate_ranking_metrics(test_df, recommend_svd, k=5, max_eval_users=10000)
metrics_10 = evaluate_ranking_metrics(test_df, recommend_svd, k=10, max_eval_users=10000)

runtime_minutes = (time.time() - start) / 60.0

all_metrics = {
    **metrics_5,
    **metrics_10,
    "explained_variance_ratio_sum": float(svd.explained_variance_ratio_.sum()),
    "runtime_minutes": float(runtime_minutes),
}

print("\nSVD Metrics")
for metric, value in all_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

# ----------------------------
# 8) Save artifacts + log to MLflow
# ----------------------------
sample_user = train_df["user_id"].iloc[0]
sample_recs = recommend_svd(sample_user, k=5)

run_payload = {
    "file_path": str(FILE_PATH),
    "train_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "user_item_shape": list(user_item.shape),
    "sample_user": int(sample_user),
    "sample_top5_recommendations": [int(x) for x in sample_recs],
    "metrics": all_metrics,
}

metrics_path = REPORT_ROOT / "svd_metrics.json"
summary_path = REPORT_ROOT / "svd_run_summary.json"
model_path = ARTIFACT_ROOT / "svd_model.pkl"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_payload, f, indent=2)

with open(model_path, "wb") as f:
    pickle.dump(
        {
            "svd": svd,
            "user_to_idx": user_to_idx,
            "item_to_idx": item_to_idx,
            "idx_to_item": idx_to_item,
            "user_factors": user_factors,
            "item_factors": item_factors,
            "seen_idx_by_user": seen_idx_by_user,
        },
        f,
    )

with mlflow.start_run(run_name="svd_recommender_baseline") as run:
    mlflow.log_param("model_type", "collaborative_filtering_svd")
    mlflow.log_param("input_file", str(FILE_PATH))
    mlflow.log_param("min_user_items", MIN_USER_ITEMS)
    mlflow.log_param("min_item_users", MIN_ITEM_USERS)
    mlflow.log_param("max_users", MAX_USERS)
    mlflow.log_param("max_items", MAX_ITEMS)
    mlflow.log_param("n_components", n_components)
    mlflow.log_param("split_type", "random_leave_one_out_after_aggregation")

    mlflow.log_param("train_rows", int(len(train_df)))
    mlflow.log_param("test_rows", int(len(test_df)))
    mlflow.log_param("train_users", int(train_df["user_id"].nunique()))
    mlflow.log_param("train_items", int(train_df["item_id"].nunique()))

    mlflow.log_metrics(all_metrics)

    mlflow.log_artifact(str(metrics_path))
    mlflow.log_artifact(str(summary_path))
    mlflow.log_artifact(str(model_path))

    mlflow.sklearn.log_model(svd, artifact_path="svd_sklearn_model")

    run_id = run.info.run_id

print("\nMLflow tracking URI:", tracking_uri)
print("MLflow run_id:", run_id)
print("Saved artifacts:")
print("-", metrics_path)
print("-", summary_path)
print("-", model_path)


C:\Users\barath\AppData\Local\anaconda3\envs\dm4ml\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Loaded file: C:\Users\barath\recomart-pipeline\data\prepared\interactions_prepared.csv
Raw shape: (2755641, 3)


Filtered shape: (30823, 3)
Unique users: 3204
Unique items: 777
Train shape: (27619, 4)
Test shape: (3204, 3)
User-item matrix shape: (3204, 777)


Chosen n_components: 20
Explained variance ratio sum: 0.489



SVD Metrics
precision_at_5: 0.0169
recall_at_5: 0.0846
ndcg_at_5: 0.0562
evaluated_users_at_5: 3204
precision_at_10: 0.0122
recall_at_10: 0.1217
ndcg_at_10: 0.0682
evaluated_users_at_10: 3204
explained_variance_ratio_sum: 0.4890
runtime_minutes: 0.0890


2026/04/29 17:05:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/29 17:06:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/29 17:06:00 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



MLflow tracking URI: file:///C:/Users/barath/recomart-pipeline/mlruns
MLflow run_id: fdea1194e151466399ac331db396b5fa
Saved artifacts:
- C:\Users\barath\recomart-pipeline\reports\svd_metrics.json
- C:\Users\barath\recomart-pipeline\reports\svd_run_summary.json
- C:\Users\barath\recomart-pipeline\models\artifacts\svd_model.pkl
